# Task 3 — GeM Gender Dataset V2 G1: Same-Canvas Foreground Mask

**Run All starts only two fits:** Gender folds 0 and 4. It reuses the exact unfiltered E6 GeM model and changes only its deterministic input view. It never trains the other Dataset V2 screens.

## 1. Mount Drive and load the Task 3 branch

Drive stores the teacher ZIP, completed causal parents, registry, and new artifacts.

In [1]:
from pathlib import Path
import os
import subprocess
import sys
import time
import zipfile

REPO_URL = "https://github.com/TrnLin/MLA2.git"
BRANCH = "fashion-analysis-and-cleanup"
REPO_DIR = Path("/content/MLA2")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_PROJECT_DIR = DRIVE_MOUNT / "MyDrive/MLA2"
DATA_ZIP = DRIVE_PROJECT_DIR / "data/task3-data.zip"
LOCAL_DATA_ZIP = Path("/content/task3-data.zip")
DRIVE_TASK_DIR = DRIVE_PROJECT_DIR / "task3"
DRIVE_REGISTRY = DRIVE_TASK_DIR / "results/runs.csv"
LOCAL_REGISTRY = REPO_DIR / "results/runs.csv"

def run_checked(command, *, cwd=None):
    command = [str(part) for part in command]
    print("$", " ".join(command), flush=True)
    return subprocess.run(command, cwd=cwd, check=True)

try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("Connect this notebook to a Google Colab GPU runtime first.") from exc

drive.mount(str(DRIVE_MOUNT), force_remount=False)
if (REPO_DIR / ".git").is_dir():
    remote_url = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=REPO_DIR, text=True
    ).strip()
    if remote_url != REPO_URL:
        raise RuntimeError(f"{REPO_DIR} belongs to a different repository: {remote_url}")
    run_checked(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
    run_checked(["git", "switch", BRANCH], cwd=REPO_DIR)
    run_checked(["git", "merge", "--ff-only", f"origin/{BRANCH}"], cwd=REPO_DIR)
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository.")
else:
    run_checked(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR])

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print("Repository ready:", REPO_DIR)
print("Branch:", BRANCH)
print("Commit:", commit)


Mounted at /content/drive
$ git clone --branch task-3-gender-usage-classification --single-branch https://github.com/TrnLin/MLA2.git /content/MLA2
Repository ready: /content/MLA2
Branch: task-3-gender-usage-classification
Commit: 56d077190108c016ad2537ecb77300b44d8f08bb


## 2. Restore teacher images on the Colab disk

Only the official teacher dataset is used. The public high-resolution collection is excluded.

In [2]:
def copy_teacher_zip_to_local_disk():
    if LOCAL_DATA_ZIP.is_file():
        try:
            with zipfile.ZipFile(LOCAL_DATA_ZIP) as existing:
                existing.infolist()
            print("Using the existing local ZIP copy:", LOCAL_DATA_ZIP)
            return
        except zipfile.BadZipFile:
            LOCAL_DATA_ZIP.unlink()

    partial = LOCAL_DATA_ZIP.with_suffix(".zip.partial")
    for attempt in range(1, 4):
        partial.unlink(missing_ok=True)
        try:
            if not DATA_ZIP.is_file():
                raise FileNotFoundError(f"Dataset archive not found: {DATA_ZIP}")
            expected_bytes = DATA_ZIP.stat().st_size
            print(
                f"Copying {expected_bytes / 1024**3:.2f} GiB from Drive "
                f"(attempt {attempt}/3)...",
                flush=True,
            )
            copied = 0
            next_report = 256 * 1024**2
            with DATA_ZIP.open("rb") as source, partial.open("wb") as target:
                while chunk := source.read(8 * 1024**2):
                    target.write(chunk)
                    copied += len(chunk)
                    if copied >= next_report:
                        print(f"  copied {copied / 1024**2:.0f} MiB", flush=True)
                        next_report += 256 * 1024**2
            if copied != expected_bytes:
                raise OSError(f"ZIP copy is incomplete: {copied} of {expected_bytes} bytes")
            partial.replace(LOCAL_DATA_ZIP)
            return
        except (OSError, FileNotFoundError) as error:
            partial.unlink(missing_ok=True)
            if attempt == 3:
                raise RuntimeError(
                    "Drive disconnected three times. Reconnect Drive, then rerun this cell."
                ) from error
            try:
                drive.flush_and_unmount()
            except Exception as unmount_error:
                print("Drive unmount warning:", unmount_error)
            drive.mount(str(DRIVE_MOUNT), force_remount=True)
            time.sleep(2)

copy_teacher_zip_to_local_disk()

teacher_dir = REPO_DIR / "data/raw/teacher"
required_files = (
    teacher_dir / "train/styles_train.csv",
    teacher_dir / "test/styles_prediction.csv",
)
image_dirs = (teacher_dir / "train/images_train", teacher_dir / "test/images_test")
image_suffixes = {".jpg", ".jpeg"}

with zipfile.ZipFile(LOCAL_DATA_ZIP) as archive:
    names = archive.namelist()
    if any(Path(name).is_absolute() or ".." in Path(name).parts for name in names):
        raise RuntimeError("The teacher archive contains an unsafe path.")
    expected_images = sum(
        name.startswith("data/raw/teacher/") and Path(name).suffix.lower() in image_suffixes
        for name in names
    )
    current_images = sum(
        path.suffix.lower() in image_suffixes for folder in image_dirs for path in folder.glob("*")
    )
    if current_images != expected_images or not all(path.is_file() for path in required_files):
        print(f"Extracting {expected_images:,} teacher images...", flush=True)
        archive.extractall(REPO_DIR)
    else:
        print("Teacher data is already extracted; skipping.")

actual_images = sum(
    path.suffix.lower() in image_suffixes for folder in image_dirs for path in folder.glob("*")
)
missing_files = [str(path) for path in required_files if not path.is_file()]
if actual_images != expected_images or missing_files:
    raise RuntimeError(
        f"Dataset check failed: expected {expected_images:,} images, found {actual_images:,}; "
        f"missing files: {missing_files}"
    )

os.chdir(REPO_DIR)
os.environ["FASHION_PROJECT_ROOT"] = str(REPO_DIR)
source_dir = str(REPO_DIR / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
(DRIVE_TASK_DIR / "results").mkdir(parents=True, exist_ok=True)
print(f"Teacher data ready: {actual_images:,} images")
print("Drive output root:", DRIVE_TASK_DIR)


Copying 0.28 GiB from Drive (attempt 1/3)...
  copied 256 MiB
Extracting 44,441 teacher images...
Teacher data ready: 44,441 images
Drive output root: /content/drive/MyDrive/MLA2/task3


## 3. Verify the frozen screen without training

The check resolves the exact five-fold causal parent, verifies the GPU and teacher files, and takes zero optimizer steps.

In [3]:
GENDER_E6_PARENT_RUN_IDS = (
    "t3_gender_e6_gem_p3_gender_smallcnngem3_f0_s2753_a8c09286451b_20260831T090059Z0bab1f",
    "t3_gender_e6_gem_p3_gender_smallcnngem3_f1_s2753_a8c09286451b_20260831T090940Z6e10b5",
    "t3_gender_e6_gem_p3_gender_smallcnngem3_f2_s2753_a8c09286451b_20260831T091823Zabb677",
    "t3_gender_e6_gem_p3_gender_smallcnngem3_f3_s2753_a8c09286451b_20260831T092710Zee7c6a",
    "t3_gender_e6_gem_p3_gender_smallcnngem3_f4_s2753_a8c09286451b_20260831T093553Z63b5fd",
)

from fashion.train.task3_dataset_v2 import check_task3_dataset_v2_setup
from fashion.train.task3_experiments import latest_completed_gender_e6_parent_run_ids

resolved_parents = latest_completed_gender_e6_parent_run_ids(output_root=DRIVE_TASK_DIR)
if resolved_parents != GENDER_E6_PARENT_RUN_IDS:
    raise RuntimeError("The resolved causal parents differ from the frozen tuple.")

preflight = check_task3_dataset_v2_setup(
    "gender_v2_foreground_mask",
    parent_run_ids=GENDER_E6_PARENT_RUN_IDS,
    output_root=DRIVE_TASK_DIR,
    root=REPO_DIR,
    device_name="cuda",
)
if preflight["input_view"] != "foreground_masked":
    raise RuntimeError("G1 must use the frozen same-canvas foreground mask.")
if preflight["training_augmentation"] != "none":
    raise RuntimeError("G1 must not add augmentation.")
if preflight["sample_weight_strategy"] != "none":
    raise RuntimeError("G1 must not add example weighting.")
print("GPU:", preflight["environment"]["gpu"])
print("Screen folds:", preflight["screen_folds"])
print("Model:", preflight["child"]["model_family"])
print("Parameters:", f'{preflight["parameter_count"]:,}')
print("Optimizer steps during check:", preflight["optimizer_steps"])


GPU: NVIDIA L4
Screen folds: [0, 4]
Model: task3_small_cnn_gem_p3
Parameters: 390,181
Optimizer steps during check: 0


## 4. Frozen hypothesis

The EDA probe improved from 0.6736 macro-F1 on full RGB to 0.6814 when border-connected near-white background pixels were masked on the same 60×80 canvas. Gender E6 also remained sensitive to canvas conditions. G1 tests whether removing that background signal reduces shortcut memorisation.

The object is not cropped, enlarged, or recentered. Pixels with all RGB channels at least 245 are removed only when connected to the border. Foreground fractions below 0.005 or above 0.95 fall back to the full frame. Validation uses the same fixed label-free view. Architecture, loss, optimiser, epoch budget, labels, and folds stay unchanged.

## 5. Train the two-fold screen

Artifacts are written directly to Drive after every epoch. A completed matching fold is reused after a disconnect.

In [4]:
from fashion.train.task3_dataset_v2 import run_task3_dataset_v2_screen
from fashion.train.task3_experiments import audit_completed_registry_rows

result = run_task3_dataset_v2_screen(
    "gender_v2_foreground_mask",
    parent_run_ids=GENDER_E6_PARENT_RUN_IDS,
    output_root=DRIVE_TASK_DIR,
    registry_path=DRIVE_REGISTRY,
    registry_mirrors=[LOCAL_REGISTRY],
    root=REPO_DIR,
    device_name="cuda",
    reuse_completed=True,
)
registry_audit = audit_completed_registry_rows(DRIVE_REGISTRY, result["fold_run_ids"])
{
    "metrics_path": result["metrics_path"],
    "folds": result["metrics"]["validation_folds"],
    "macro_f1": result["metrics"]["macro_f1"],
    
    "mean_train_validation_gap": result["metrics"]["mean_final_train_validation_gap"],
    "registry_audit": registry_audit,
}


[task3] preparing target=gender fold=0: train=26,220 (before selection=26,220), validation=6,553
[task3] fitting fold-training RGB statistics for target=gender fold=0
[task3] RGB statistics ready for target=gender fold=0
[task3] registered t3_gender_v2_g1_foreground_mask_gender_smallcnngem3_f0_s2753_473d124afa95_20260904T131930Z29e258; the first optimiser step may now run
[task3] target=gender fold=0 epoch=1/30 train_loss=0.5533 train_macro_f1=0.4881 validation_loss=0.5778 validation_macro_f1=0.3550
[task3] target=gender fold=0 epoch=2/30 train_loss=0.4103 train_macro_f1=0.6348 validation_loss=0.4433 validation_macro_f1=0.6534
[task3] target=gender fold=0 epoch=3/30 train_loss=0.3533 train_macro_f1=0.6988 validation_loss=0.3873 validation_macro_f1=0.6224
[task3] target=gender fold=0 epoch=4/30 train_loss=0.3146 train_macro_f1=0.7335 validation_loss=0.3531 validation_macro_f1=0.6909
[task3] target=gender fold=0 epoch=5/30 train_loss=0.2808 train_macro_f1=0.7667 validation_loss=0.4725 va

{'metrics_path': '/content/drive/MyDrive/MLA2/task3/experiments/t3_gender_v2_g1_foreground_mask/gender/aggregate/metrics.json',
 'folds': [0, 4],
 'macro_f1': 0.7295373647213312,
 'mean_train_validation_gap': 0.27029785236159953,
 'registry_audit': {'registry_path': '/content/drive/MyDrive/MLA2/task3/results/runs.csv',
  'run_ids': ['t3_gender_v2_g1_foreground_mask_gender_smallcnngem3_f0_s2753_473d124afa95_20260904T131930Z29e258',
   't3_gender_v2_g1_foreground_mask_gender_smallcnngem3_f4_s2753_473d124afa95_20260904T133343Zf2d8ba'],
  'completed_rows': 2,
  'ready': True}}

## 6. Confirm the saved evidence

This cell only summarizes the finished folds. It does not promote the model.

In [5]:
import pandas as pd

display(pd.DataFrame([
    {
        "screen": "gender_v2_foreground_mask",
        "folds": result["metrics"]["validation_folds"],
        "macro_f1": result["metrics"]["macro_f1"],
        "fold_sd": result["metrics"]["fold_macro_f1_sample_sd"],
        "mean_train_validation_gap": result["metrics"]["mean_final_train_validation_gap"],
        "metrics_path": result["metrics_path"],
    }
]))
print("Stop here. This two-fold result must be analysed before any five-fold promotion.")


,screen,folds,macro_f1,fold_sd,mean_train_validation_gap,metrics_path
0,gender_v2_foreground_mask,"[0, 4]",0.729537,0.007996,0.270298,/content/drive/MyDrive/MLA2/task3/experiments/...


Stop here. This two-fold result must be analysed before any five-fold promotion.
